<h1>YodAI: An AI-based Search Tool</h1>

<p>We have been tasked with the following things until the IRD doc has been approved. As such, we will work on the pipeline development based on the datasets and metadata we currently have.</p>

<h3>To-do</h3>

<ol>
<li><b>Begin Pipeline Development Using Metadata</b>

Start building the framework for:
<ul>
<li>Data ingestion</li>
<li>Cleaning</li>
<li>Querying</li>
<li>Modeling</li>
<li>Using metadata or synthetic data.</li>
</ul>
</li>


<li><b>Design Data Cleaning Pipeline</b>

Prepare scripts to:
<ul>
<li>Standardize variables</li>
<li>Handle missing data</li>
<li>Normalize categorical fields</li>
<li>Structure patient-visit relationships.</li>
</ul>
</li>


<li><b>Build Queryable Database Structure</b>

Design the structure for:
<ul>
<li>Patient table
<li>Visit table
<li>Labs table
<li>Complications table.</li>
</ul>
</li>


<li><b>Create Synthetic Dataset for Testing</b>

Generate simulated data that matches the metadata schema so the team can:
<ul>
<li>Test pipelines</li>
<li>Debug models</li>
<li>Develop queries.</li>
</ul>
</li>


<li><b>Define Modeling Goals</b>

Prepare analytic tasks such as:
<ul>
<li>Predict diabetes complications</li>
<li>Identify risk clusters</li>
<li>Analyze treatment patterns.</li>
</ul>
</li>


<li><b>Literature Review</b>

Students should begin:
<ul>
<li>Literature review on diabetes registries</li>
<li>ML models for diabetes outcomes</li>
<li>Clinical risk prediction.</li>
</ul>
</li>


<li><b>Prepare for Data Integration</b>

Once data arrives:
<ul>
<li>Plug dataset into pipeline</li>
<li>Run cleaning</li>
<li>Conduct exploratory analysis.</li>
</ul>
</li>
</ol>

<h2>0. Importing libraries</h2>

In [83]:
import pandas as pd
import numpy as np
import re
import os

<h2>1. Exploring the dataset</h2>

In [84]:
df_metadata = pd.read_csv('dmc_data_structure_metadata.csv')
df_sample = pd.read_excel('dmc_data_sample.xlsx')

print('M E T A D A T A')
display(df_metadata.head())
print('S A M P L E')
display(df_sample.head())

M E T A D A T A


,Column_Name,Data_Type,Missing_Count,Missing_Percentage,Unique_Values
0,Pin,character,0,0.00,88132
1,FirstName,character,9983,2.57,8307
2,LastName,character,9983,2.57,5535
3,FatherName,character,9984,2.57,14981
4,Birthday,character,9983,2.57,16


S A M P L E


,Pin,Gender,RegistrationDate,MartialStatus,VisitDate,Expr2,Expr3,id,DiagnosticScore,DiagnosticStatus,...,DMOnsetMonth,DMOnsetYear,DMOnsetInvestigation,IsConsultantAvailable,OtherReasonToReferCons,Reason,RetinaImageData,HB,UricAcid,TLC
0,310351,Female,2015-07-15,Married,2015-07-15,NaN,DM,2,NaN,DM,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,310351,Female,2015-07-15,Married,2015-07-15,NaN,DM,3,NaN,DM,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,310358,Male,2015-07-16,NaN,2015-07-16,NaN,DM,4,NaN,DM,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,310360,Female,2015-07-16,NaN,2015-07-16,NaN,DM,8,NaN,DM,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,310364,Male,2015-07-16,NaN,2015-07-16,NaN,DM,10,NaN,DM,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<h3>!!!!!!!!!!!!! ALL THE CODE BETWEEN THE SET BLOCK WILL BE DELETED !!!!!!!!!!!!!</h3>
<hr>

In [85]:
metadata_columns = df_metadata['Column_Name'].tolist()
sample_columns = df_sample.keys().tolist()

print(len(metadata_columns), metadata_columns)
print(len(sample_columns), sample_columns)

199 ['Pin', 'FirstName', 'LastName', 'FatherName', 'Birthday', 'Gender', 'RegistrationDate', 'PhoneNo1', 'MartialStatus', 'VisitDate', 'DoctorName', 'Expr2', 'Expr3', 'id...14', 'DiagnosticScore', 'DiagnosticStatus', 'EtiologicalScore', 'EtiologicalStatus', 'FunctionalStatus', 'FunctionalStatus2', 'Complications', 'Referalreasonagos', 'ReasonForReferals', 'InsulinR', 'InsulinRFullDose', 'InsulinN', 'InsulinNFullDose', 'Insulin7030', 'Insulin7030FullDose', 'ACarboseFullDose', 'ACarbose', 'NextVisit', 'PatientTreatmentPlan', 'CurrentSuggestedInsulin', 'CurrentSuggestedOHG', 'DietCalories', 'Instruction2', 'MetforminDose', 'MetforminFullDose', 'SitagliptinDose', 'SitagliptinFullDose', 'GlimiprideDose', 'GlimiprideFullDose', 'PioglitazoneDose', 'PioglitazoneFullDose', 'GITDrug1', 'GITDrug1FullDose', 'GITDrug2', 'GITDrug2FullDose', 'GITDrug3', 'GITDrug3FullDose', 'NeuropathyPainKiller1', 'NeuropathyPainKiller1Dose', 'NeuropathyPainKiller2', 'NeuropathyPainKiller2Dose', 'NeuropathyPainKiller

<h3>Comparing the two datasets we currently have</h3>

In [86]:
# Comparing sample columns against metadata columns
missing_metadata_columns = []
for column in metadata_columns:
    if column not in sample_columns:
        missing_metadata_columns.append(column)

print('Missing columns in sample data:', missing_metadata_columns)
print('Total missing columns:', len(missing_metadata_columns))

Missing columns in sample data: ['FirstName', 'LastName', 'FatherName', 'Birthday', 'PhoneNo1', 'DoctorName', 'id...14', 'RefferedBy', 'Consultant', 'PlannedBy', 'id...159', 'seq']
Total missing columns: 12


In [87]:
# Comparing metadata columns against sample columns
missing_sample_columns = []
for column in sample_columns:
    if column not in metadata_columns:
        missing_sample_columns.append(column)

print('Missing columns in sample data:', missing_sample_columns)
print('Total missing columns:', len(missing_sample_columns))

Missing columns in sample data: ['id', 'id.1']
Total missing columns: 2


<h2>Information about current datasets</h2>

We currently have two datasets - DMC metadata, and Sample DMC data. Below is the information about each of them:

<ul>
<li>
<b>DMC metadata</b> (dmc_data_structure_metadata.csv)

This metadata for the original dataset has 199 values with 4 feature traits. It contains information about the column names of the original dataset, and some very basic information as show below:

<ul>
<li><b>Data_Type:</b> All the elements are "character" except "Expr4" and "seq". This represents the original dataset having primarily character-based features (other than these two features).</li>
<li><b>Missing_Count:</b> Shows missing values in each column</li>
<li><b>Missing_Percentage:</b> Shows what percentage of the entire column is missing</li>
<li><b>Unique_Values:</b> Shows number of unique elements in each column</li>
</ul>
</li>


<li>
<b>Sample DMC data</b> (dmc_data_sample.xlsx)

This dataset contains a total of 29 unique values, over 189 features. Roughly half the features are either completely NULL values or completely duplicated values - only a few of them have 1-2 non-NULL/non-duplicate values.

Other than that, this dataset contains 2 extra columns "id" and "Id.1". Moreover, it has 10 features absent that are otherwise present in the DMC metadata. Note that since this is just a sample dataset, you don't need to merge them.
</li>
</ul>

<hr>

<h2>2. Data cleaning</h2>

Given the overall structure of both the datasets, I would recommend doing the following for clarity:

<ol>
<li>Remove features that have only-NULL or a single value for all elements</li>
<li>Standardize all non-numerical feature values</li>
<li>Alter column values appropriately (set null to 0 or "N/A")</li>
<li>See which features are purely unique to set the primary key accordingly. You can start with 'Expr4' and 'seq' first since they're the only numerical columns</li>
</ol>


<h3>1. Removing NULL values and single-value columns</h3>

In [88]:
cols_info = []
for col in df_sample.columns:
    if df_sample[col].isna().all():
        cols_info.append((col, 'NULL'))
    
    if df_sample[col].nunique(dropna=True) == 1 and not df_sample[col].isna().any():
        cols_info.append((col, 'SINGLE_VALUE'))

cols_info.sort(key=lambda x: x[1])
for i in range(len(cols_info)):
    print(cols_info[i])

('Expr2', 'NULL')
('DiagnosticScore', 'NULL')
('EtiologicalScore', 'NULL')
('FunctionalStatus', 'NULL')
('FunctionalStatus2', 'NULL')
('Referalreasonagos', 'NULL')
('GlimiprideDose', 'NULL')
('GlimiprideFullDose', 'NULL')
('PioglitazoneDose', 'NULL')
('PioglitazoneFullDose', 'NULL')
('NeuropathyPainKiller2', 'NULL')
('NeuropathyPainKiller2Dose', 'NULL')
('NeuropathyPainKiller3', 'NULL')
('NeuropathyPainKiller3Dose', 'NULL')
('MedicationTable', 'NULL')
('Somogyii', 'NULL')
('CounselingAdvice', 'NULL')
('PatientCategory', 'NULL')
('LipidLoweringDrug2Dose', 'NULL')
('LipidLoweringDrug2FullDose', 'NULL')
('BetaBlockerDose', 'NULL')
('BetaBlockerFullDose', 'NULL')
('CCBDeureticDose', 'NULL')
('CCBDeureticFullDose', 'NULL')
('LasorideDose', 'NULL')
('LasorideFullDose', 'NULL')
('Combinaion', 'NULL')
('AllowOrNotPlanFasting', 'NULL')
('PrePostAvgValue', 'NULL')
('PrePostScoreAvg', 'NULL')
('RBSavgVaue', 'NULL')
('RevisitTime', 'NULL')
('Referal1', 'NULL')
('Investigation1', 'NULL')
('AfterHow

In [89]:
# ------------------------------------- RUN ONCE ONLY -------------------------------------
print('Column length before dropping NULLs:', len(df_sample.columns))
df_sample.drop([col for col, col_type in cols_info if col_type == 'NULL'], axis=1, inplace=True)
print('Column length after dropping NULLs:', len(df_sample.columns))

print('-'*60)

print('Column length before dropping single value columns:', len(df_sample.columns))
df_sample.drop([col for col, col_type in cols_info if col_type == 'SINGLE_VALUE'], axis=1, inplace=True)
print('Column length after dropping single value columns:', len(df_sample.columns))

Column length before dropping NULLs: 189
Column length after dropping NULLs: 100
------------------------------------------------------------
Column length before dropping single value columns: 100
Column length after dropping single value columns: 96


**COMMENTS:**

<p>It is seen that our dataset size has essentially been halved after removing columns with only null values or only single-values. From those filtered, we still have multiple columns which have just a few meaningful values, all else being null. We will have to deal with those columns manually to retain meaning.</p>

<p>We have set up a custom threshold value to ensure that the data we use is useful. If the column consists of more than 80% NULL values, we will drop the column. But to verify that we are not removing any important information, we will have to validate these manually first.
</p>


---------- COMPARISON WITH <i>dmc_data_structure_metadata</i> ----------

We will have to compare the sample_data with the metadata to see which columns to remove for now.

In [90]:
# Change null values to '-'
for col in df_sample.columns:
    df_sample[col] = df_sample[col].fillna('-')

display(df_sample)

,Pin,Gender,RegistrationDate,MartialStatus,VisitDate,id,EtiologicalStatus,Complications,ReasonForReferals,InsulinR,...,Creatinine,Proteinuria,PusCell,TotalCholesterol,LDL,HDL,Triglycerides,ALT,HBsAg,AntiHCV
0,310351,Female,2015-07-15,Married,2015-07-15,2,Mixed,-,I do not agree with the choice of medication,Inj. Insuman R (کھانے سے آدھا گھنٹا پہلے),...,-,-,-,-,-,-,-,-,-,-
1,310351,Female,2015-07-15,Married,2015-07-15,3,Mixed,-,I do not agree with the choice of medication,Inj. Insuman R (کھانے سے آدھا گھنٹا پہلے),...,-,-,-,-,-,-,-,-,-,-
2,310358,Male,2015-07-16,-,2015-07-16,4,Mixed,-,-,-,...,-,-,-,-,-,-,-,-,-,-
3,310360,Female,2015-07-16,-,2015-07-16,8,Insulin Resistant,Neuropathy,-,-,...,-,-,-,-,-,-,-,-,-,-
4,310364,Male,2015-07-16,-,2015-07-16,10,Insulin Resistant,Neuropathy,Automatic plan not generated,-,...,-,-,-,-,-,-,-,-,-,-
5,310368,Male,2015-07-16,-,2015-07-16,19,Insulin Requiring,Neuropathy,I do not agree with the etiological status,Inj. Insuman R (کھانے سے آدھا گھنٹا پہلے),...,0.8,N/A,2025-01-05 00:00:00,189.0,80.0,40.0,195.0,17.0,N/A,N/A
6,310375,Male,2015-07-16,-,2015-07-16,26,Mixed,-,-,-,...,-,-,-,-,-,-,-,-,-,-
7,310377,Female,2015-07-16,-,2015-07-16,24,Mixed,-,I do not agree with the diagnostic status,Inj. Insuman R (کھانے سے آدھا گھنٹا پہلے),...,1.8,-,-,-,-,-,-,-,-,-
8,310377,Female,2015-07-16,-,2015-07-22,24,Mixed,-,I do not agree with the diagnostic status,Inj. Insuman R (کھانے سے آدھا گھنٹا پہلے),...,-,-,-,-,-,-,-,-,-,-
9,310378,Male,2015-07-22,-,2015-07-02,27,Mixed,Nephropathy,-,Inj. Insuman R (کھانے سے آدھا گھنٹا پہلے),...,1.8,-,-,-,-,-,-,-,-,-


In [91]:
FILTER_THRESHOLD = 0.2*len(df_sample)
filter_cols = []
for col in df_sample.columns:
    if df_sample[col].unique().shape[0] <= 3:            # Change with FILTER_THRESHOLD
        print(f'\n{col} ---> {df_sample[col].nunique()} unique values: {df_sample[col].unique()}')
        filter_cols.append(col)

print(len(filter_cols), '\n', filter_cols)


Gender ---> 2 unique values: <StringArray>
['Female', 'Male']
Length: 2, dtype: str

MartialStatus ---> 2 unique values: <StringArray>
['Married', '-']
Length: 2, dtype: str

EtiologicalStatus ---> 3 unique values: <StringArray>
['Mixed', 'Insulin Resistant', 'Insulin Requiring']
Length: 3, dtype: str

InsulinR ---> 2 unique values: <StringArray>
['Inj. Insuman R    (کھانے سے آدھا گھنٹا پہلے)', '-']
Length: 2, dtype: str

InsulinN ---> 2 unique values: <StringArray>
['Inj.Insuman N    (رات۱۱ بجے)', '-']
Length: 2, dtype: str

Insulin7030 ---> 2 unique values: <StringArray>
['-', 'Inj. Insuman 75/25    (کھانے سے آدھا گھنٹا پہلے)']
Length: 2, dtype: str

ACarboseFullDose ---> 2 unique values: <StringArray>
['1+1+1', '-']
Length: 2, dtype: str

ACarbose ---> 2 unique values: <StringArray>
['Tab. Glucobay 25 mg    (کھانے کے پہلے نوالے کے ساتھ)', '-']
Length: 2, dtype: str

PatientTreatmentPlan ---> 2 unique values: <StringArray>
['Mixed', 'InsulinPlan']
Length: 2, dtype: str

CurrentSugge

In [92]:
filter_dict = {}
for col in df_sample.columns:
    unique_vals = df_sample[col].unique()
    if len(unique_vals) < 4:
        non_dash_count = (df_sample[col] != '-').sum()
        pct_non_dash = non_dash_count / len(df_sample) * 100

        if non_dash_count < 4:
            category = 'Discardable'
        elif pct_non_dash > 50:
            category = 'Sufficient'
        else:
            category = 'Moderate'

        filter_dict[col] = (list(unique_vals), non_dash_count, category)

filter_dict = dict(sorted(filter_dict.items(), key=lambda item: item[1][2]))

In [93]:
category_list = ['Discardable', 'Moderate', 'Sufficient']
for category in category_list:
    print(category)
    for col, info in filter_dict.items():
        if info[2] == category:
            print(f"  - {col}:\n      unique={info[0]}\n      non_dash_count={info[1]}")
    print()

Discardable
  - ACarboseFullDose:
      unique=['1+1+1', '-']
      non_dash_count=1
  - ACarbose:
      unique=['Tab. Glucobay 25 mg    (کھانے کے پہلے نوالے کے ساتھ)', '-']
      non_dash_count=1
  - SitagliptinDose:
      unique=['-', 'Tab. Trevia    (کھانے سے آدھا گھنٹا پہلے)']
      non_dash_count=1
  - SitagliptinFullDose:
      unique=['-', '50+0+0']
      non_dash_count=1
  - ARBDose:
      unique=['-', 'Tab. Losartan 50mg']
      non_dash_count=1
  - ARBFullDose:
      unique=['-', '0+0+50']
      non_dash_count=1
  - CCBlockerDose:
      unique=['Tab. Amlodipine', '-']
      non_dash_count=3
  - CCBlockerFullDose:
      unique=['0+5+0', '-']
      non_dash_count=3
  - AlphaBlockerDose:
      unique=['Tab. Prazosin', '-']
      non_dash_count=2
  - AlphaBlockerFullDose:
      unique=['0+0+1', '-']
      non_dash_count=2
  - ARBDeureticDose:
      unique=['-', 'Tab. Xavor Diu']
      non_dash_count=2
  - ARBDeureticFullDose:
      unique=['-', '1+0+0']
      non_dash_count=2
  -

In [94]:
df_sample.drop([col for col, info in filter_dict.items() if info[2] == 'Discardable'], axis=1, inplace=True)

In [95]:
display(df_sample)

,Pin,Gender,RegistrationDate,MartialStatus,VisitDate,id,EtiologicalStatus,Complications,ReasonForReferals,InsulinR,...,Goiter,Pallor,Jaundice,SOFluidOverload,SOSeriousIllnessOtherThanDiabetes,Sensation,Calluses,Deformity,id.1,Creatinine
0,310351,Female,2015-07-15,Married,2015-07-15,2,Mixed,-,I do not agree with the choice of medication,Inj. Insuman R (کھانے سے آدھا گھنٹا پہلے),...,-,-,-,-,-,-,-,-,330466,-
1,310351,Female,2015-07-15,Married,2015-07-15,3,Mixed,-,I do not agree with the choice of medication,Inj. Insuman R (کھانے سے آدھا گھنٹا پہلے),...,-,-,-,-,-,-,-,-,330466,-
2,310358,Male,2015-07-16,-,2015-07-16,4,Mixed,-,-,-,...,No,No,No,No,-,Normal,Yes,No,330468,-
3,310360,Female,2015-07-16,-,2015-07-16,8,Insulin Resistant,Neuropathy,-,-,...,No,No,No,No,Yes,Diminished,No,No,330470,-
4,310364,Male,2015-07-16,-,2015-07-16,10,Insulin Resistant,Neuropathy,Automatic plan not generated,-,...,No,Yes,No,Yes,-,Diminished,No,No,330474,-
5,310368,Male,2015-07-16,-,2015-07-16,19,Insulin Requiring,Neuropathy,I do not agree with the etiological status,Inj. Insuman R (کھانے سے آدھا گھنٹا پہلے),...,No,No,No,No,No,Diminished,No,No,330478,0.8
6,310375,Male,2015-07-16,-,2015-07-16,26,Mixed,-,-,-,...,No,No,No,No,No,-,-,-,330484,-
7,310377,Female,2015-07-16,-,2015-07-16,24,Mixed,-,I do not agree with the diagnostic status,Inj. Insuman R (کھانے سے آدھا گھنٹا پہلے),...,-,-,-,-,-,-,-,-,330486,1.8
8,310377,Female,2015-07-16,-,2015-07-22,24,Mixed,-,I do not agree with the diagnostic status,Inj. Insuman R (کھانے سے آدھا گھنٹا پہلے),...,-,-,-,-,-,-,-,-,330487,-
9,310378,Male,2015-07-22,-,2015-07-02,27,Mixed,Nephropathy,-,Inj. Insuman R (کھانے سے آدھا گھنٹا پہلے),...,-,-,-,-,-,-,-,-,330488,1.8
